In [1]:
import psycopg2
from pyspark.sql import SparkSession

spark = SparkSession.builder .appName("Postgres") .config("spark.jars", "/home/namuna-acharya/jars/postgresql-42.5.6.jar") .getOrCreate()

25/04/30 14:08:10 WARN Utils: Your hostname, namunaacharya resolves to a loopback address: 127.0.1.1; using 192.168.1.190 instead (on interface wlx60fb0067d846)
25/04/30 14:08:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/04/30 14:08:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
spark

In [3]:
port = 5432
database = "postgres"
host = "localhost"
user = "postgres"
password = "sql123"

In [4]:
jdbc_url = "jdbc:postgresql://localhost:5432/postgres"
jdbc_properties = {
    "user": user,
    "password": password,
    "driver": "org.postgresql.Driver"
}

In [5]:
connection = psycopg2.connect(
    host= host,   
    dbname = database,
    user= user,        
    password=password,
    port= port)

cursor = connection.cursor()

In [6]:
provider_table = """
CREATE TABLE IF NOT EXISTS provider_table(
    provider_group_id BIGINT,
    npi BIGINT,
    tin_type SMALLINT,
    tin TEXT
);
"""

cursor.execute(provider_table)
connection.commit()

In [7]:
in_network_table= """
CREATE TABLE IF NOT EXISTS in_network_table(
    billing_code TEXT,
    billing_code_type TEXT,
    negotiation_arrangement TEXT,    
    provider_group_id BIGINT,     
    billing_class TEXT,
    billing_code_modifier TEXT[],
    negotiated_rate DOUBLE PRECISION,    
    negotiated_type TEXT,
    service_code INTEGER[]
);
"""
cursor.execute(in_network_table)
connection.commit()

In [8]:
provider = spark.read.parquet("files/provider_data.parquet")
in_network = spark.read.parquet("files/rate_data.parquet")

In [9]:
provider.write.jdbc(url=jdbc_url,table="provider_table",mode="append", properties=jdbc_properties)

In [10]:
in_network.write.jdbc(url=jdbc_url,table="in_network_table",mode="append", properties=jdbc_properties)

In [11]:
cursor.close()
connection.close()